## Ground Truth

In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [4]:
documents = documents_llm
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [9]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [6]:
import json

user_prompt = json.dumps(doc)

In [23]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [ ]:
# We used "openai_client.responses.create" before
# To include a custom output formatgting we need to use "responses.parse" instead
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

result = response.output_parsed

print(result)

questions=['Can I still join the course if I found it late?', 'If I join now, can I still get the certificate?', 'Do I need to finish the project before submissions close to get a certificate?', 'Is it okay to start the course after it already began?', 'What’s the deadline for submitting the project if I want the certificate?']


In [ ]:
print(result.questions)

['Can I still join the course if I found it late?', 'If I join now, can I still get the certificate?', 'Do I need to finish the project before submissions close to get a certificate?', 'Is it okay to start the course after it already began?', 'What’s the deadline for submitting the project if I want the certificate?']


In [ ]:
from evaluation_utils import llm_structured

# The same as above but from a reusable package
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — is it too late to join, or can I still enroll now?', 'If I join after the course has already started, can I still get a certificate somehow?', 'What’s the deadline for turning in the project if I want the certificate?', 'Do late joiners miss out on the certificate, or is it still possible if I submit the project in time?', 'Can I start the course now and still be eligible for certification, or is that only for people who joined earlier?']


In [26]:
usage.input_tokens, usage.output_tokens

(207, 115)

In [27]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

{'input_cost': 0.00015525,
 'output_cost': 0.0005175000000000001,
 'total_cost': 0.0006727500000000001}

In [ ]:
# Ground truth records - questions generated from a given documents (labelled with an id)
# Now we can use that questions in search to validate if the right documents are found

records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — is it too late to join, or can I still enroll now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, can I still get a certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for turning in the project if I want the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do late joiners miss out on the certificate, or is it still possible if I submit the project in time?',
  'document': '74eb249bbf'},
 {'question': 'Can I start the course now and still be eligible for certification, or is that only for people who joined earlier?',
  'document': '74eb249bbf'}]

### Ground Truth - full set

In [1]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [10]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [11]:
# Parallelize LLM API calls

from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [12]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [13]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08720699999999999

In [15]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08720699999999999

In [17]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth

,question,document
0,I just found this course late — can I still si...,74eb249bbf
1,Am I allowed to join the course after it alrea...,74eb249bbf
2,"If I start the course now, is there still a ch...",74eb249bbf
3,What do I need to do to be eligible for a cert...,74eb249bbf
4,"Is late enrollment okay, or is the project dea...",74eb249bbf
...,...,...
560,Why am I getting a 401 Client Error when tryin...,4b30b918bc
561,What should I do if my requests install keeps ...,4b30b918bc
562,Is there a way to force pip to install request...,4b30b918bc
563,"Could this error mean my API key is wrong, or ...",4b30b918bc


In [19]:
# Save ground truth to csv for later use

df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)